<a href="https://colab.research.google.com/github/Gr1lledChee5e/LLM/blob/main/Final_Project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install miceforest (and LightGBM dependency) if not already installed
# !pip install --quiet miceforest lightgbm

# Load Packages

In [ ]:
!pip install pregress

In [ ]:
!pip install --quiet miceforest lightgbm # Install miceforest (and LightGBM dependency) if not already installed

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pregress as pr

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, Normalizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, RocCurveDisplay

import miceforest as mf

warnings.filterwarnings("ignore")
RANDOM_STATE = 123
DATA_DIR = Path('.')
plt.style.use('seaborn-v0_8')

ModuleNotFoundError: No module named 'miceforest'

# Load Data

In [ ]:
train_path = DATA_DIR / 'train_data.csv'
test_path = DATA_DIR / 'test_data.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Convert target to numeric 0/1
target = 'Churn'
train_df[target] = train_df[target].map({'Yes': 1, 'No': 0})

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
train_df.head()


# Basic Data Checks

In [ ]:
print('Target distribution:')
print(train_df[target].value_counts(normalize=True))

missing = train_df.isnull().mean().sort_values(ascending=False)
missing


In [ ]:
# train_df['ContractType'].value_counts(dropna=False)
train_df.columns

In [ ]:
f_cols = []
for i in range(1, 21):
    f_cols.append(f'F{i}')
pr.boxplot(train_df[f_cols])

In [ ]:
f_cols.append('Churn')
# pr.plots('Churn ~ .', train_df[f_cols])

In [ ]:
pr.plot_cor(train_df[f_cols])

# Feature Engineering

In [ ]:
reference_date = pd.Timestamp('2025-01-01')
for df in [train_df, test_df]:
    df['DateOfServiceStart'] = pd.to_datetime(df['DateOfServiceStart'])
    df['AccountAgeYears'] = (reference_date - df['DateOfServiceStart']).dt.days / 365
    # df['PricePerGB'] = df['MonthlyCharges'] / df['DataUsagePerMonth']
    df['ContractType_Monthly'] = (df['ContractType'] == 'Monthly').astype(int)
    df['ServiceType_FiberOptic'] = (df['ServiceType'] == 'Fiber Optic').astype(int)
    df['Autopay'] = df['Autopay'].map({'Yes': 1, 'No': 0})
    # df = pd.get_dummies(df, columns=['PaymentMethod'], drop_first=False)#.drop(columns=['DateOfServiceStart', 'ContractType', 'ServiceType'])
    # df.drop(columns=['DateOfServiceStart', 'ContractType', 'ServiceType'])

train_df = pd.get_dummies(train_df, columns=['PaymentMethod'], drop_first=False)
test_df = pd.get_dummies(test_df, columns=['PaymentMethod'], drop_first=False)
train_features = train_df.drop(columns=[target])
test_features = test_df.copy()

train_features, test_features = train_features.align(test_features, join='left', axis=1, fill_value=0)

id_col = 'ID'
drop_cols = [id_col, 'DateOfServiceStart', 'ContractType', 'ServiceType']
feature_cols = [c for c in train_features.columns if c not in drop_cols]

X_full = train_features[feature_cols].copy()
y_full = train_df[target]
X_test = test_features[feature_cols].copy()

# Split labeled data into train/holdout before CV
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=RANDOM_STATE
)

print('Feature count after dummies:', len(feature_cols))
missing_frac = X_full.isnull().mean()
print('Columns with missing values (numeric only):')
print(missing_frac[missing_frac > 0].sort_values(ascending=False))
print('Train/val shapes:', X_train_raw.shape, X_val_raw.shape)


In [ ]:
X_train_raw.columns

In [ ]:
X_train_raw.head()

# Preprocessing

In [ ]:
SCALER_NAME = 'minmax'           # choose: 'standard', 'minmax', 'normalize'
IMPUTER_NAME = 'knn'  # options: 'median', 'knn', 'iterative', 'miceforest_numeric'
MICE_ITERATIONS = 15
MICE_ESTIMATORS = 200

scalers = {
    'standard': StandardScaler(),
    'minmax': MinMaxScaler(),
    'normalize': Normalizer(),
}

from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

imputers = {
    'median': SimpleImputer(strategy='median'),
    'knn': KNNImputer(n_neighbors=10),
    'iterative': IterativeImputer(random_state=RANDOM_STATE, max_iter=15),
}


class MiceForestImputerNumeric(BaseEstimator, TransformerMixin):
    """MICE imputation on numeric columns only."""

    def __init__(self, random_state: int = RANDOM_STATE, iterations: int = 5, estimators: int = 50):
        self.random_state = random_state
        self.iterations = iterations
        self.estimators = estimators

    def fit(self, X, y=None):  # noqa: N802
        X_df = pd.DataFrame(X).reset_index(drop=True)
        self.columns_ = X_df.columns.tolist()

        self.kernel_ = mf.ImputationKernel(
            data=X_df,
            random_state=self.random_state,
            mean_match_candidates=5,
        )
        self.kernel_.mice(
            iterations=self.iterations,
            verbose=False,
            num_iterations=self.estimators,
        )
        return self

    def transform(self, X):  # noqa: N802
        X_df = pd.DataFrame(X).reset_index(drop=True)
        imputed = self.kernel_.impute_new_data(
            new_data=X_df,
            iterations=self.iterations,
            verbose=False,
        )
        completed = imputed.complete_data(dataset=0)
        completed = completed[self.columns_]
        return completed.values


if IMPUTER_NAME == 'miceforest_numeric':
    imputer = MiceForestImputerNumeric(
        random_state=RANDOM_STATE,
        iterations=MICE_ITERATIONS,
        estimators=MICE_ESTIMATORS,
    )
else:
    imputer = imputers[IMPUTER_NAME]

scaler = scalers[SCALER_NAME]
preprocess = Pipeline([
    ('imputer', imputer),
    ('scaler', scaler),
])

# Fit once on training split, reuse across CV/holdout/test
preprocess.fit(X_train_raw, y_train)
X_train_processed = preprocess.transform(X_train_raw)
X_val_processed = preprocess.transform(X_val_raw)
X_full_processed = preprocess.transform(X_full)
X_test_processed = preprocess.transform(X_test)

print(f"Numeric imputer: {IMPUTER_NAME}; Scaler: {SCALER_NAME}")
if IMPUTER_NAME == 'miceforest_numeric':
    print(f"MICE iterations: {MICE_ITERATIONS}; estimators: {MICE_ESTIMATORS}")
print(f'Processed shapes -> train/val: {X_train_processed.shape}, {X_val_processed.shape} || full/test: {X_full_processed.shape}, {X_test_processed.shape}')


# Cross-Validation Setup

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Shared results store and search helper
results = []
best_models = {}

def run_search(name, estimator, param_grid, X_data=None, y_data=None):
    """Run GridSearchCV with ROC-AUC on preprocessed data, store best model and results."""
    data_X = X_train_processed if X_data is None else X_data
    data_y = y_train if y_data is None else y_data

    search = GridSearchCV(
        estimator,
        param_grid=param_grid,
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1,
        refit=True,
        return_train_score=False,
    )
    search.fit(data_X, data_y)

    global results, best_models
    results = [r for r in results if r['model'] != name]
    if name in best_models:
        best_models.pop(name)

    best_models[name] = search
    results.append({
        'model': name,
        'best_auc': search.best_score_,
        'best_params': search.best_params_,
    })

    print(f"{name}: best AUC = {search.best_score_:.4f}")
    print('best params:', search.best_params_)
    return search


# Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=2000, solver='lbfgs')
log_reg_params = {'C': [0.001, 0.01, 0.1, 0.25, 0.5, 1.0, 5]}
log_reg_search = run_search('log_reg', log_reg, log_reg_params)


# Lasso (L1 Logistic Regression)

In [ ]:
lasso = LogisticRegression(penalty='l1', solver='saga', max_iter=4000)
lasso_params = {'C': [0.05, 0.5, 1.0, 5.0]}
lasso_search = run_search('lasso', lasso, lasso_params)


# Ridge (L2 Logistic Regression)

In [ ]:
ridge = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=2000)
ridge_params = {'C': [0.1, 1.0, 10.0]}
ridge_search = run_search('ridge', ridge, ridge_params)


# Elastic Net Logistic Regression

In [ ]:
elastic = LogisticRegression(penalty='elasticnet', solver='saga', max_iter=4000)
elastic_params = {
    'C': np.linspace(0, 0.5, 11),
    'l1_ratio': np.linspace(0, 1, 11),
}
elastic_search = run_search('elastic_net', elastic, elastic_params)


# Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

# best params: {'max_depth': 10, 'max_features': 0.8, 'min_samples_leaf': 10, 'min_samples_split': 2, 'n_estimators': 200}
rf_params = {
    'n_estimators': [600],
    'max_depth': [None],
    'min_samples_leaf': [50, 100, 150],
    'max_features': [0.4, 0.5],
    'class_weight': ['balanced']
}

rf_search = run_search('random_forest', rf, rf_params)


In [ ]:
# rf_bal = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')

# rf_bal_search = run_search('random_forest_balanced', rf_bal, rf_params)


# K-Nearest Neighbors

In [ ]:
knn = KNeighborsClassifier()
knn_params = {
    'p': [1, 2],
    'n_neighbors': [50, 100, 150],
    'weights': ['uniform', 'distance']
}
knn_search = run_search('knn', knn, knn_params)

# Results Summary (run after any models above)

In [ ]:
results_df = (
    pd.DataFrame(results)
    .sort_values('best_auc', ascending=False)
    .reset_index(drop=True)
)
results_df

# lasso	0.689670 {'model__C': 0.5} (standard, knn=5)
# elastic_net	0.692482 {'model__C': 0.1, 'model__l1_ratio': 0.2} (minmax, knn=5)


In [ ]:
model = results_df.iloc[0]['model']
auc = round(results_df.iloc[0]['best_auc'], 4)
params = results_df.iloc[0]['best_params']
imputer = IMPUTER_NAME
scaler = SCALER_NAME


print(f'{auc} -> {model} | {scaler}, {imputer} | {params}')

0.6891 -> lasso | standard, median | {'model__C': 0.05}
0.6924 -> elastic_net | minmax, median | {'model__C': 0.1, 'model__l1_ratio': 0.2}
0.6345 -> random_forest | normalize, median | {'model__max_depth': 10, 'model__min_samples_leaf': 5, 'model__n_estimators': 200}

0.6895 -> lasso | standard, knn | {'model__C': 0.05}
0.6924 -> elastic_net | minmax, knn | {'model__C': 0.1, 'model__l1_ratio': 0.2}

0.69 -> lasso | standard, iterative | {'model__C': 0.05}
0.6926 -> elastic_net | minmax, iterative | {'model__C': 0.1, 'model__l1_ratio': 0.2}

# Feature Importance (models with coefficients/importances)

Top features for models that expose coefficients or feature importances.

In [ ]:
from IPython.display import display

if not results:
    raise ValueError('Run at least one model section to populate results.')

feature_names = X_full.columns.tolist()
importance_tables = {}

for name, search in best_models.items():
    est = search.best_estimator_
    if hasattr(est, 'coef_'):
        coef_vec = est.coef_[0]
        imp = np.abs(coef_vec)
        importance_tables[name] = (
            pd.DataFrame({
                'feature': feature_names,
                'importance': imp,
                'coefficient': coef_vec,
            })
            .sort_values('importance', ascending=False)
            .reset_index(drop=True)
        )
    elif hasattr(est, 'feature_importances_'):
        imp = est.feature_importances_
        importance_tables[name] = (
            pd.DataFrame({
                'feature': feature_names,
                'importance': imp,
            })
            .sort_values('importance', ascending=False)
            .reset_index(drop=True)
        )
    else:
        continue

if not importance_tables:
    print('No models with importances/coefficients available. Run compatible models first.')
else:
    for name, df_imp in importance_tables.items():
        top = df_imp.head(15)
        print(f"{name} top features:")
        # display(top)
        plt.figure(figsize=(8, 5))
        plt.barh(top['feature'], top['importance'], color='#1f77b4')
        plt.gca().invert_yaxis()
        plt.xlabel('Importance (abs coef or importance)')
        plt.title(f'{name} feature importance (top 15)')
        plt.tight_layout()
        plt.show()


# Validation ROC Curve (best available model)

In [ ]:
if not results:
    raise ValueError('Run at least one model section to populate results.')

results_df = (
    pd.DataFrame(results)
    .sort_values('best_auc', ascending=False)
    .reset_index(drop=True)
)

best_model_name = results_df.loc[0, 'model']
best_search = best_models[best_model_name]
best_model = best_search.best_estimator_

best_model.fit(X_train_processed, y_train)
val_pred = best_model.predict_proba(X_val_processed)[:, 1]
val_auc = roc_auc_score(y_val, val_pred)
print('Using best model:', best_model_name)
print('Holdout ROC AUC:', round(val_auc, 4))

fig, ax = plt.subplots()
RocCurveDisplay.from_predictions(y_val, val_pred, ax=ax)
ax.set_title('Validation ROC curve')
plt.show()


# Train Full Model and Create Submission

In [ ]:
if not results:
    raise ValueError('Run at least one model section to populate results.')

results_df = (
    pd.DataFrame(results)
    .sort_values('best_auc', ascending=False)
    .reset_index(drop=True)
)

best_model_name = results_df.loc[0, 'model']
best_model = best_models[best_model_name].best_estimator_

best_model.fit(X_processed, y)

test_pred = best_model.predict_proba(X_test_processed)[:, 1]

submission = pd.DataFrame({
    'ID': test_df[id_col],
    'ChurnProb': test_pred,
})

submission_path = Path('best_model_1.csv')
submission.to_csv(submission_path, index=False)
print('Saved:', submission_path.resolve())
submission.head()


```markdown
## Summary for GitHub

This notebook demonstrates a machine learning pipeline for churn prediction. It covers data loading, preprocessing, feature engineering, model training, evaluation, and submission.

### Key Steps:

1.  **Data Loading**: Initial training and test datasets (`train_data.csv` and `test_data.csv`) are loaded into pandas DataFrames. The target variable 'Churn' is converted to a numerical format (0/1).

2.  **Feature Engineering**: New features are created, such as `AccountAgeYears` (derived from `DateOfServiceStart`), and categorical features like `ContractType` and `ServiceType` are one-hot encoded (`ContractType_Monthly`, `ServiceType_FiberOptic`). `PaymentMethod` is also one-hot encoded.

3.  **Preprocessing**: A flexible preprocessing pipeline is defined, consisting of imputation and scaling. Options include:
    *   **Imputers**: `median`, `knn` (K-Nearest Neighbors Imputer), `iterative` (Iterative Imputer), and a custom `miceforest_numeric` (MICE imputation for numerical columns).
    *   **Scalers**: `standard` (StandardScaler), `minmax` (MinMaxScaler), `normalize` (Normalizer).
    The pipeline is fitted on the training data and then used to transform all datasets.

4.  **Model Training and Cross-Validation**: Several classification models are trained and evaluated using `GridSearchCV` with `StratifiedKFold` (5 splits) and `roc_auc` as the scoring metric. Models explored include:
    *   Logistic Regression
    *   Lasso (L1 Logistic Regression)
    *   Ridge (L2 Logistic Regression)
    *   Elastic Net Logistic Regression
    *   Random Forest Classifier
    *   K-Nearest Neighbors Classifier

5.  **Results Summary**: The best performing models and their parameters are summarized based on their cross-validation AUC scores.

6.  **Feature Importance/Coefficients**: For models that support it (e.g., Logistic Regression variants, Random Forest), feature importances or coefficients are extracted and visualized to understand key drivers of churn.

7.  **Validation ROC Curve**: The ROC curve for the best model on the holdout validation set is plotted to visually assess its performance.

8.  **Final Model Training and Submission**: The best model (selected based on validation AUC) is retrained on the full training dataset (`X_full_processed`, `y_full`). Predictions are then made on the unseen test data, and a submission file (`best_model_1.csv`) is generated with `ID` and `ChurnProb` columns.

This notebook provides a comprehensive approach to building a churn prediction model, with emphasis on robust preprocessing and model selection.
```